In [2]:
import numpy as np
import glob
from PIL import Image
from scipy import spatial

In [10]:
def montage(input_pic, tile_photos, output_file, tile_size):

    #Get all tiles
    tile_paths = []
    for file in glob.glob(tile_photos):
        tile_paths.append(file)

    # Import and resize all tiles
    tiles = []
    for path in tile_paths:
        tile = Image.open(path)
        tile = tile.resize(tile_size)
        tile = tile.convert("RGB")  #要不然有一些的image mode變成P，結果沒辦法用rgb下去算
        tiles.append(tile)

    # mode ref: https://pillow.readthedocs.io/en/stable/handbook/concepts.html#concept-modes

    # Calculate dominant color
    colors = []
    for tile in tiles:
        mean_color = np.array(tile).mean(axis=0).mean(axis=0)
        colors.append(mean_color)

    # Pixelate (resize) main photo
    main_photo = Image.open(input_pic)

    width = int(np.round(main_photo.size[0] / tile_size[0]))
    height = int(np.round(main_photo.size[1] / tile_size[1]))

    resized_photo = main_photo.resize((width, height))

    # Find closest tile photo for every pixel
    # Create a KDTree
    tree = spatial.KDTree(colors) ## 會報錯 --> 發現是colors裡面有些會變"一個數字"，才知道是原圖有些的 image mode 是rgb, rgba, p --> 是 p 會變成一個數字

    # Empty integer array to store indices of tiles
    closest_tiles = np.zeros((width, height), dtype=np.uint32)

    for i in range(width):
        for j in range(height):
            pixel = resized_photo.getpixel((i, j))  # Getthe pixel color at (i, j)
            closest = tree.query(pixel)             # Returns (distance, index)
            closest_tiles[i, j] = closest[1]        # We only need the index


    # Create an output image
    output = Image.new('RGB', main_photo.size)

    # Draw tiles
    for i in range(width):
        for j in range(height):
            # Offset of tile
            x, y = i*tile_size[0], j*tile_size[1]
            # Index of tile
            index = closest_tiles[i, j]
            # Draw tile
            output.paste(tiles[index], (x, y))

    output.save(output_file)

montage("assets\\hw2_pic1.jpg","mosaic-master\\mosaic-master\\dataset\\*","output.jpg", (10, 10))
#output要用jpg檔案

c:\Users\soarb\.conda\envs\jupyter_server\lib\site-packages\PIL\Image.py:970: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
